In [8]:
import rasterio
import numpy as np
import joblib

# Load the pre-trained models
rf_model = joblib.load('../data/processed/rf_best_model.pkl')
qrf_model = joblib.load('../data/processed/qrf_best_model.pkl')

# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = rf_model.predict(input_raster_valid)

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

# Predict using the loaded QRF model
quantiles = np.arange(0.01, 1.01, 0.01).tolist()  # Convert to list
qrf_output_valid = qrf_model.predict(input_raster_valid, quantiles=quantiles)

# Create an output array for QRF and fill with NaNs
qrf_output_raster = np.full((height * width, qrf_output_valid.shape[1]), np.nan)
qrf_output_raster[valid_mask] = qrf_output_valid
qrf_output_raster = qrf_output_raster.reshape(height, width, qrf_output_valid.shape[1])

# Update the profile for the QRF output raster
qrf_profile = profile.copy()
qrf_profile.update(count=qrf_output_valid.shape[1])

# Write the QRF output raster
output_qrf_file = '../data/processed/qrf_predictions_africa.tif'
with rasterio.open(output_qrf_file, 'w', **qrf_profile) as dst:
    for i in range(qrf_output_valid.shape[1]):
        dst.write(qrf_output_raster[:, :, i], i + 1)

C:\Users\DHOUGNI\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\DHOUGNI\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestQuantileRegressor was fitted with feature names
  warnings.warn(
